# Phase 1 analysis from saved results

Recomputes every verdict, table and figure in the README from the files under `experiments/`
(run outputs downloaded from the training volume: per-position NLL, minting records, evaluation curves) and
`results/`. No GPU and no training: this notebook only reads saved results.

Inputs: `experiments/modal-m3/runs/*` (primary canonical set, Protocol 05), `results/r0b/`, `results/growth/`,
`results/headcost/`, `results/probe/`, `data/tokens/` (token arrays for byte counts).

In [1]:
import json, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
RUNS = [str(ROOT / "experiments" / "modal-m3" / "runs")]
OUT = ROOT / "results" / "final"

## 1. Recompute the verdicts (two-level paired bootstrap, Protocol 03)

In [2]:
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "final_analysis.py"), "--runs", *RUNS,
                    "--prefix", "m3", "--split", "test", "--out", str(OUT)], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-3000:] or r.stderr[-3000:])
A = json.load(open(OUT / "analysis_m3_test.json"))

{
 "arms": {
  "Dense": [
   1.442741495665308,
   0.00482133276348673
  ],
  "KAS-0": [
   1.939758849337148,
   0.0003078839521506449
  ],
  "KAS-G": [
   1.5138022752228641,
   0.008570235972703344
  ],
  "KAS-G-shuf": [
   1.5278782600012273,
   0.0028267478735242157
  ],
  "KAS-P": [
   1.630780687984679,
   0.00013835484706525847
  ],
  "KAS-U16": [
   1.4612712354119408,
   0.0009924352522090452
  ],
  "KAS-U16 (c_zero)": [
   1.5829896785226873,
   0.00042851710831559586
  ]
 },
 "verdicts": {
  "H1": "SUPPORTED",
  "H2": "SUPPORTED",
  "H3": "INCONCLUSIVE",
  "H4": "REJECTED"
 },
 "fairness": {
  "distinct_train_orders": 1,
  "distinct_total_steps": [
   1526
  ],
  "distinct_data_manifests": 1,
  "seeds_per_arm": {
   "Dense": [
    1,
    2
   ],
   "KAS-0": [
    1,
    2
   ],
   "KAS-G": [
    1,
    2
   ],
   "KAS-G-shuf": [
    1,
    2
   ],
   "KAS-P": [
    1,
    2
   ],
   "KAS-U16 (c_zero)": [
    1,
    2
   ],
   "KAS-U16": [
    1,
    2
   ]
  },
  "distinct_

## 2. Bits per byte on the test split (seed means and spread)

In [3]:
for arm, v in sorted(A["arms"].items(), key=lambda kv: kv[1]["bpb_mean"]):
    print(f"{arm:12s} bpb {v['bpb_mean']:.4f}  seeds {['%.4f' % x for x in v['bpb_per_seed']]}  "
          f"spread {v['seed_spread'] if v['seed_spread'] is None else round(v['seed_spread'], 4)}  "
          f"head params {v['head_params']:,}  V-dependent {v['head_V_dependent']:,}")
print("fairness:", A["fairness"])

Dense        bpb 1.4427  seeds ['1.4452', '1.4403']  spread 0.0048  head params 29,308,416  V-dependent 29,308,416
KAS-U16      bpb 1.4613  seeds ['1.4608', '1.4618']  spread 0.001  head params 5,175,660  V-dependent 973,131
KAS-G        bpb 1.5138  seeds ['1.5181', '1.5095']  spread 0.0086  head params 4,954,418  V-dependent 0
KAS-G-shuf   bpb 1.5279  seeds ['1.5293', '1.5265']  spread 0.0028  head params 4,954,418  V-dependent 0
KAS-U16 (c_zero) bpb 1.5830  seeds ['1.5828', '1.5832']  spread 0.0004  head params 5,175,660  V-dependent 973,131
KAS-P        bpb 1.6308  seeds ['1.6307', '1.6308']  spread 0.0001  head params 4,194,337  V-dependent 0
KAS-0        bpb 1.9398  seeds ['1.9396', '1.9399']  spread 0.0003  head params 4,194,337  V-dependent 0
fairness: {'distinct_train_orders': 1, 'distinct_total_steps': [1526], 'distinct_data_manifests': 1, 'seeds_per_arm': {'Dense': [1, 2], 'KAS-0': [1, 2], 'KAS-G': [1, 2], 'KAS-G-shuf': [1, 2], 'KAS-P': [1, 2], 'KAS-U16 (c_zero)': [1, 2], 'KA

## 3. Hypotheses

In [4]:
for h in ("H1", "H2", "H3", "H4"):
    v = A.get(h, {})
    print(h, "->", v.get("verdict"))
print(json.dumps({k: A.get("H1", {}).get(k) for k in ("rho", "denominator_P_minus_U", "outcome_band")}, indent=1))
print(json.dumps({k: A.get("H2", {}).get(k) for k in ("kasu_rule_selected_on_dev", "kasg_rule_selected_on_dev",
                                                       "delta_regret_bits", "delta_leak_bpb")}, indent=1))
print(json.dumps({k: A.get("H4", {}).get(k) for k in ("delta_shuf_minus_kasu", "delta_spell_kasg_minus_shuf")}, indent=1))

H1 -> SUPPORTED
H2 -> SUPPORTED
H3 -> INCONCLUSIVE
H4 -> REJECTED
{
 "rho": {
  "point": 0.6900996433317973,
  "ci95": [
   0.6612564261495546,
   0.718476662773882
  ],
  "n_boot": 2000
 },
 "denominator_P_minus_U": {
  "point": 0.16950945257273808,
  "ci95": [
   0.1678272070485076,
   0.17111933792169068
  ],
  "n_boot": 2000
 },
 "outcome_band": "success"
}
{
 "kasu_rule_selected_on_dev": "inherit",
 "kasg_rule_selected_on_dev": "inherit",
 "delta_regret_bits": {
  "point": -0.37650660021733406,
  "ci95": [
   -0.5433385140149443,
   -0.20044256607831046
  ],
  "n_sites": 2110,
  "n_windows": 748
 },
 "delta_leak_bpb": {
  "point": -0.0025491592190320803,
  "ci95": [
   -0.0026875058766384382,
   -0.00241803345162073
  ],
  "n_boot": 2000
 }
}
{
 "delta_shuf_minus_kasu": {
  "point": -0.277150588416476,
  "ci95": [
   -0.42352102042235573,
   -0.13331363320708509
  ],
  "n_sites": 2110,
  "n_windows": 748
 },
 "delta_spell_kasg_minus_shuf": {
  "point": -0.09935601180085807,
  "ci9

## 4. Minting table (test; mean regret in bits per site, beats-prefix, median rank, leak)

In [5]:
t = A["minting_table"]["test"]
for arm, rules in t.items():
    print(arm, "sites:", rules.get("_n_sites"))
    for name, v in rules.items():
        if name.startswith("_"):
            continue
        print(f"   {name:14s} regret {v['mean_regret_bits']:8.3f}  beats {v['beats_prefix']:.3f}  "
              f"median rank {v['median_rank']:8.0f}  leak {v['leak_bpb']:.5f}")

Dense sites: 2110
   mean           regret    7.697  beats 0.001  median rank    26668  leak 0.00014
   inherit        regret    0.766  beats 0.000  median rank     1306  leak 0.00057
KAS-0 sites: 2110
   bytes          regret   -1.704  beats 0.238  median rank     2872  leak 0.00328
KAS-G sites: 2110
   floor          regret   18.437  beats 0.003  median rank    54586  leak 0.00005
   zero           regret    9.201  beats 0.047  median rank    31206  leak 0.00234
   count          regret    9.201  beats 0.044  median rank    30693  leak 0.00202
   inherit        regret    0.670  beats 0.104  median rank     2296  leak 0.00074
   gen_floor      regret   10.105  beats 0.000  median rank    32002  leak 0.00000
   gen_zero       regret    0.833  beats 0.134  median rank     2572  leak 0.00067
   gen_count      regret    0.839  beats 0.140  median rank     2476  leak 0.00055
   gen_inherit    regret    0.839  beats 0.140  median rank     2476  leak 0.00055
   count_cal      regret   11.298

## 5. Where the gap lives: NLL(Dense) − NLL(KAS) by training frequency

In [6]:
for arm, rows in A.get("H3", {}).get("buckets", {}).items():
    print(arm)
    for r in rows:
        if r.get("n_targets"):
            print(f"   {r['bucket']:16s} n={r['n_targets']:8d}  d={r['d_nats']:+.4f}  CI {r['ci95'][0]:+.4f} .. {r['ci95'][1]:+.4f}")

KAS-U16
   [0,1)            n=       1  d=-1.1057  CI -2.1895 .. -0.0219
   [1,10)           n=      17  d=-0.5983  CI -1.3995 .. +0.1400
   [10,100)         n=    1191  d=+0.4706  CI +0.1483 .. +0.8221
   [100,1000)       n=   66564  d=-0.1079  CI -0.1590 .. -0.0586
   [1000,10000)     n=  213347  d=-0.0987  CI -0.1246 .. -0.0719
   [10000,100000)   n=  300202  d=-0.0985  CI -0.1209 .. -0.0768
   [100000,1000000) n=  200537  d=-0.0747  CI -0.0903 .. -0.0591
   [1000000,inf)    n=  348442  d=+0.0117  CI +0.0036 .. +0.0194
KAS-P
   [0,1)            n=       1  d=-3.8900  CI -4.8193 .. -2.9608
   [1,10)           n=      17  d=+0.2397  CI -0.9083 .. +1.2510
   [10,100)         n=    1191  d=+1.4295  CI +1.0528 .. +1.8035
   [100,1000)       n=   66564  d=-1.0262  CI -1.0914 .. -0.9620
   [1000,10000)     n=  213347  d=-1.1499  CI -1.1870 .. -1.1139
   [10000,100000)   n=  300202  d=-0.8893  CI -0.9162 .. -0.8605
   [100000,1000000) n=  200537  d=-0.4650  CI -0.4839 .. -0.4461
   [1000000

## 6. Figures

The image outputs are not stored in this notebook; the figures it writes are in `figures/`.

In [7]:
subprocess.run([sys.executable, str(ROOT / "scripts" / "make_figures.py"), "--analysis", str(OUT / "analysis_m3_test.json")], cwd=ROOT, check=True)
from IPython.display import Image, display
for f in sorted((ROOT / "figures").glob("fig*.png")):
    print(f.name); display(Image(filename=str(f), width=700))

fig1_gap_vs_tokens.png


fig2_head_params_vs_bpb.png


fig3_frequency_buckets.png


fig4_minting_regret_leak_net.png


fig5_vocab_growth.png


fig6_r0b_predictability.png


fig7_head_cost_vs_V.png
